# Aula 01 — IA aplicada + Hello Modelo (Iris)


**Disciplina:** Inteligência Artificial  
**Professor:** Marcelo Batista  


## 🎯 Objetivos

1) Entender o ambiente do Google Colab CPU/GPU  
2) Manipular dados com Pandas utilizando dataset - Iris  
3) Treinar seu primeiro modelo de ML (baseline)  
4) Entender o fluxo sagrado: **Dados → Treino → Teste**  
5) Aprender o **pipeline reprodutível** (pré-processamento + modelo)
6) Exercícios

## Teoria


- IA aplicada = resolver problema real com dados + métricas.
- Modelo = função parametrizada que generaliza padrões.
- Treino ajusta parâmetros para reduzir erros.
- Teste mede generalização em dados não vistos.
- Pipeline evita bagunça e aumenta a reprodutibilidade.


## ⚙️ Como alterar o tipo de ambiente de execução (CPU/GPU/TPU) no Google Colab

- Colab pode iniciar em CPU.
- Para trocar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.
- Trocar CPU↔GPU pode reiniciar o runtime (você perde variáveis e roda tudo de novo).
- Nesta aula (Scikit-Learn), GPU **não é obrigatória** — mas vamos checar.

## Configuração do ambiente (CPU vs GPU)


- No Colab, às vezes você está em **CPU** por padrão.
- Se estiver em GPU, o comando `nvidia-smi` funciona.
- Se não estiver, ele não existe e dá erro — isso é normal.


In [ ]:
#Checar GPU de forma “à prova de erro”
import shutil, subprocess, textwrap


def try_nvidia_smi():
    if shutil.which("nvidia-smi") is None:
        print("GPU NVIDIA não detectada (runtime provavelmente em CPU).")
        print("No Colab: Runtime/Ambiente de execução → Alterar tipo de hardware → GPU.")
        return
    out = subprocess.check_output(["nvidia-smi"], text=True)
    print(out)


try_nvidia_smi()


Mon Mar 16 17:25:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bibliotecas que vamos usar


- **Pandas**: “Excel do Python”
- **Seaborn**: datasets prontos + gráficos
- **Scikit-Learn**: ML clássico (split, modelos, métricas, pipelines)

In [ ]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")

## Dataset Iris (o “Hello World” da IA)

Aqui o objetivo é **entender o fluxo**:
- carregar dados
- olhar a tabela
- separar X e y
- dividir treino/teste
- treinar
- medir

### Existem quase 300 espécies diferentes de Iris já descobertas. Para o nosso objetivo nesta aula, vamos utilizar 3 espécies:

- Setosa
- Versicolor
- Virginica

![](https://content.codecademy.com/programs/machine-learning/k-means/iris.svg)

![](https://miro.medium.com/max/3500/1*f6KbPXwksAliMIsibFyGJw.png)


In [ ]:
df = sns.load_dataset("iris")
display(df.head())
print("shape:", df.shape)
display(df["species"].value_counts())

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


shape: (150, 5)


,count
species,
setosa,50
versicolor,50
virginica,50


## 🎯 Problema (Iris)
- **X (features):** medidas da flor (sépalas/pétalas)
- **y (target):** espécie (`setosa`, `versicolor`, `virginica`)

In [ ]:
# Separar X e y
X = df.drop("species", axis=1)
y = df["species"]

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

Formato de X: (150, 4)
Formato de y: (150,)


In [ ]:
# Estatística básica + contagem por classe
display(df.describe())
display(df["species"].value_counts())

,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


,count
species,
setosa,50
versicolor,50
virginica,50


X = df.drop("species", axis=1)

- df.drop(): Este é um método do Pandas usado para remover linhas ou colunas de um DataFrame.

- "species": Aqui, você está especificando o nome da coluna que deseja remover. No contexto do dataset Iris, "species" é a coluna que contém a espécie da flor, que é a variável que queremos prever (o "alvo" ou "target").

- axis=1: Este argumento é crucial. Ele indica que a operação de drop deve ser aplicada às colunas do DataFrame.

- Se fosse axis=0, a operação seria aplicada às linhas. Ao usar axis=1, estamos dizendo ao Pandas para "remover a coluna 'species'".

- Resultado: A variável X (que representa as features ou características) receberá um novo DataFrame contendo todas as colunas originais do df exceto a coluna "species".

## ✅ Checkpoint (EDA rápido)
Olhe:
- quantidade de linhas/colunas (`shape`)
- distribuição das classes (`value_counts`)
- se tem valores ausentes (vamos checar no Exercício A)

### Treino vs Teste (conceito crítico)

- Treino X: onde o modelo aprende padrões.
- Teste y: onde avaliamos se ele generaliza.
- Se eu avaliar no treino, posso “me enganar” (overfitting).

In [ ]:
X

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [ ]:
y

,species
0,setosa
1,setosa
2,setosa
3,setosa
4,setosa
...,...
145,virginica
146,virginica
147,virginica
148,virginica


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)
acc = accuracy_score(y_test, pred)

print(f"Acurácia (Iris, split único): {acc:.4f}")

Acurácia (Iris, split único): 1.0000


Em resumo, a DecisionTreeClassifier é um algoritmo poderoso para encontrar limites de decisão claros, e o dataset Iris possui características que permitem que esses limites sejam encontrados com alta precisão, resultando em uma classificação perfeita para o conjunto de teste específico gerado.

É importante notar que 100% de acurácia é raro em problemas do mundo real e, em contextos mais complexos, poderia ser um sinal de overfitting (o modelo "decorou" os dados de treino e não generaliza bem), mas no Iris, é mais um reflexo da simplicidade e separabilidade dos dados

In [ ]:
print(f"Acurácia (Iris): {acc*100:.2f}%")

Acurácia (Iris): 100.00%


Natureza do Dataset Iris: O dataset Iris é um conjunto de dados clássico e relativamente "fácil" para muitos algoritmos de classificação.

As classes de flores (setosa, versicolor, virginica) são, em grande parte, linearmente separáveis ou separáveis por regras simples baseadas nas características das pétalas e sépalas.

A espécie "setosa", por exemplo, é quase sempre perfeitamente distinguível das outras.

## ⚠️ Importante: 100% de acurácia nem sempre é “bom”
- Pode ser split “sortudo”
- Pode ser overfitting (modelo decorou padrões do treino)
- Pode ser vazamento de dados (leakage) em problemas reais
✅ Vamos confirmar com validação cruzada na sequência.

### Teste “mundo real”

In [ ]:
nova_flor = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=X.columns)
print("Previsão para nova flor:", modelo.predict(nova_flor)[0])

Previsão para nova flor: setosa


A **DecisionTreeClassifier** (Árvore de Decisão para Classificação) é um algoritmo de Machine Learning supervisionado usado para resolver problemas de classificação.

Como funciona:

- Ela constrói um modelo em forma de árvore, onde cada nó interno representa um teste em uma característica (feature).
- Cada ramo representa o resultado desse teste.
- Cada nó folha (terminal) representa uma decisão de classe.

Características principais:

- Intuitiva e interpretável: A lógica de decisão pode ser visualizada e entendida facilmente.
- Versátil: Lida bem com dados numéricos e categóricos.
- Base para outros modelos: É o componente fundamental de algoritmos mais avançados, como Random Forests e Gradient Boosting.
- Em essência, ela "aprende" uma série de regras "se-então" a partir dos dados para prever a classe de novas observações.

# 🧪 Exercício A — Entendendo os dados (Iris)

## Faça:
1) Mostre:
- `df.shape`
- `df.info()`
- `df.isna().sum()`

2) Mostre:
- `df["species"].value_counts()`

3) Escreva **2 linhas** explicando:
- **o que é X (features)** e **o que é y (target)** nesse dataset Iris

## Evidência mínima (saída):
- prints/outputs do `shape`, `isna().sum()` e `value_counts()`

# 🧪 Exercício B — Seu primeiro baseline (Iris)

## Objetivo
Comparar como a **divisão treino/teste** impacta a acurácia do modelo.

## Faça (duas comparações)

### 1) Comparar `test_size` (mantendo `random_state=42`)
Rode o mesmo modelo duas vezes:
- `test_size=0.2`
- `test_size=0.3`

✅ **Você deve obter 2 acurácias** (uma para cada `test_size`).

### 2) Comparar `random_state` (mantendo `test_size=0.2`)
Rode o mesmo modelo duas vezes:
- `random_state=42`
- `random_state=7`

✅ **Você deve obter 2 acurácias** (uma para cada `random_state`).

## Regras
- Use o mesmo modelo: `DecisionTreeClassifier(random_state=42)`
- Use o dataset `iris`
- Não altere as features nem o target (X e y)

## Evidência mínima (saída)
- **2 acurácias (test_size diferente)** + **2 acurácias (random_state diferente)**

# Resoluções dos Exercício A e B

In [ ]:
# Exercício A — Entendendo os dados (Iris)

import pandas as pd

print("1) Dimensão do dataset (linhas, colunas):")
print("df.shape =", df.shape)

print("\n2) Info do DataFrame (tipos + não-nulos):")
df.info()

print("\n3) Quantidade de valores ausentes por coluna:")
display(df.isna().sum())

print("\n4) Contagem por classe (species):")
display(df["species"].value_counts())

# 5) Resposta (2 linhas) — preencha aqui:
resposta_1 = "X (features) é ..."
resposta_2 = "y (target) é ..."

print("\n5) Sua explicação (2 linhas):")
print("-", resposta_1)
print("-", resposta_2)

1) Dimensão do dataset (linhas, colunas):
df.shape = (150, 5)

2) Info do DataFrame (tipos + não-nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB

3) Quantidade de valores ausentes por coluna:


,0
sepal_length,0
sepal_width,0
petal_length,0
petal_width,0
species,0



4) Contagem por classe (species):


,count
species,
setosa,50
versicolor,50
virginica,50



5) Sua explicação (2 linhas):
- X (features) é ...
- y (target) é ...


In [ ]:
# ✅ SOLUÇÃO — Exercício B (Iris): baseline + comparação de splits

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Pré-requisito: df já existe (df = sns.load_dataset("iris"))
X = df.drop("species", axis=1)
y = df["species"]

def treinar_e_avaliar(test_size, random_state):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    modelo = DecisionTreeClassifier(random_state=42)  # modelo fixo (só mudamos o split)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, pred)
    return acc

# 1) Comparar test_size (random_state fixo)
acc_ts_02 = treinar_e_avaliar(test_size=0.2, random_state=42)
acc_ts_03 = treinar_e_avaliar(test_size=0.3, random_state=42)

# 2) Comparar random_state (test_size fixo)
acc_rs_42 = treinar_e_avaliar(test_size=0.2, random_state=42)
acc_rs_07 = treinar_e_avaliar(test_size=0.2, random_state=7)

print("=== SOLUÇÃO: Exercício B (Iris) ===")
print("\n[Parte 1] Comparação de test_size (random_state = 42)")
print(f"Acurácia com test_size=0.2: {acc_ts_02:.4f}")
print(f"Acurácia com test_size=0.3: {acc_ts_03:.4f}")

print("\n[Parte 2] Comparação de random_state (test_size = 0.2)")
print(f"Acurácia com random_state=42: {acc_rs_42:.4f}")
print(f"Acurácia com random_state=7:  {acc_rs_07:.4f}")

=== SOLUÇÃO: Exercício B (Iris) ===

[Parte 1] Comparação de test_size (random_state = 42)
Acurácia com test_size=0.2: 1.0000
Acurácia com test_size=0.3: 1.0000

[Parte 2] Comparação de random_state (test_size = 0.2)
Acurácia com random_state=42: 1.0000
Acurácia com random_state=7:  0.9000


## 🧠 Por que a acurácia mudou quando alterei `random_state` (e por que `test_size` às vezes não muda)?

A diferença apareceu porque **você não mudou o modelo**, e sim **mudou o “sorteio” de quais exemplos caem no treino e quais caem no teste**.  
Em datasets pequenos como o **Iris (150 amostras)**, esse sorteio pode alterar bastante o “nível de dificuldade” do conjunto de teste.

---

### 1) O que muda quando você troca `random_state`?
`random_state` define a *semente* do embaralhamento (o “jeito” como os dados são separados).  
Ou seja, com seeds diferentes, você pode acabar com:

- um teste “fácil” (poucos exemplos ambíguos / na fronteira entre classes)
- um teste “difícil” (mais exemplos parecidos entre classes)

✅ No seu caso:
- `random_state=42` gerou um conjunto de teste que, por sorte, ficou **muito fácil** para a Árvore de Decisão → **1.0000**
- `random_state=7` gerou um conjunto de teste **mais difícil** (provavelmente com mais casos na fronteira, especialmente entre *versicolor* e *virginica*) → **0.9000**

> Analogia: você estudou o mesmo conteúdo, mas a **prova** mudou. Uma prova pode cair exatamente o que você domina; outra pode focar nos pontos mais confusos.

---

### 2) E o `test_size`? Por que às vezes não muda nada?
Quando você muda `test_size`, você muda:

- o quanto o modelo tem para **aprender** (treino)
- o quanto você usa para **avaliar** (teste)

Mas o Iris é um dataset relativamente “separável”, e a Árvore de Decisão costuma performar muito bem.  
Então pode acontecer de, mesmo com menos dados no treino (70% em vez de 80%), o modelo ainda aprender regras suficientemente boas — e o teste (por sorte) não conter muitos exemplos difíceis.

Por isso você pode ver **1.0000 em ambos os test_size** em alguns splits.

---

### 3) Por que 100% de acurácia pode acontecer no Iris?
O Iris é um dataset “didático”:
- *setosa* é quase sempre fácil de separar
- a maior confusão costuma ocorrer entre *versicolor* e *virginica*

Se o conjunto de teste tiver poucos casos “ambíguos” dessas duas classes, a acurácia pode ficar perfeita.

---

### 4) O que essa diferença ensina (lição de ML)
✅ **Lição 1: Acurácia de um único split é frágil**  
Um único sorteio treino/teste pode gerar uma estimativa otimista (100%) ou mais realista (90%).

✅ **Lição 2: Por isso usamos validação cruzada (cross-validation)**  
Ela avalia o modelo em vários splits diferentes e tira uma média, reduzindo o efeito “sorte/azar” da divisão.

---

### Resumo para anotar (1 frase)
A acurácia mudou porque `random_state` e `test_size` alteram quais exemplos vão para treino e teste; em bases pequenas isso muda a dificuldade do teste, então um único split pode enganar — validação cruzada é mais confiável.

In [ ]:
# ✅ Célula (Código) — Validação Cruzada (Cross-Validation) no Iris
# Objetivo: medir a performance de forma mais robusta do que um único train/test split

import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

# Pré-requisito: df já existe (df = sns.load_dataset("iris"))
X = df.drop("species", axis=1)
y = df["species"]

# Modelo (o mesmo do exercício)
modelo = DecisionTreeClassifier(random_state=42)

# K-Fold estratificado: mantém proporção das classes em cada dobra
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    modelo, X, y,
    cv=cv,
    scoring="accuracy"
)

print("=== Validação Cruzada (5-fold Stratified) ===")
print("Acurácias por fold:", np.round(scores, 4))
print(f"Média: {scores.mean():.4f}")
print(f"Desvio padrão: {scores.std():.4f}")

print("\nInterpretação:")
print("- A média resume o desempenho esperado do modelo.")
print("- O desvio padrão indica o quanto o resultado varia conforme o split (instabilidade).")

=== Validação Cruzada (5-fold Stratified) ===
Acurácias por fold: [1.     0.9667 0.9333 0.9667 0.9   ]
Média: 0.9533
Desvio padrão: 0.0340

Interpretação:
- A média resume o desempenho esperado do modelo.
- O desvio padrão indica o quanto o resultado varia conforme o split (instabilidade).


A validação cruzada (5-fold) mostra que a acurácia do modelo varia conforme o split, ficando entre **0.90 e 1.00**, ou seja, um único treino/teste pode dar um resultado “sortudo” (1.00) ou mais difícil (0.90).  
Em média, o desempenho esperado é **0.9533** com **desvio padrão 0.0340**, indicando boa performance geral, mas com uma variação real causada pela forma como os dados são divididos.

## Por que **100% de acurácia** (quase sempre) **não é “bom”** — é **suspeito** 👀

Em ML aplicado, **100%** costuma acender uma luz amarela, porque frequentemente significa **problema de avaliação**, e não “modelo genial”.  
Aqui estão os principais motivos (bem didático):

---

### 1) 🎯 Você pode estar medindo no lugar errado (ou do jeito errado)

**a) Avaliar no treino (erro clássico)**  
Se você mede no **mesmo conjunto que treinou**, o modelo pode “decorar” os dados (**overfitting**) e dar 100%.

> **Regra de ouro:** *Treino serve para aprender; teste serve para julgar.*

**b) Split “sortudo”**  
Em bases pequenas, um `random_state` pode gerar um teste “fácil” e sair 100% **por sorte**.  
No Iris isso acontece bastante porque o dataset é “limpo” e algumas classes são bem separáveis.

✅ **Por isso existe validação cruzada:** ela repete vários splits e mostra a variação real.

---

